## Imports

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

## Data Load

In [3]:
df = pd.read_csv("../model/placementdata.csv")
df.head()

,StudentID,CGPA,Internships,Projects,Workshops/Certifications,AptitudeTestScore,SoftSkillsRating,ExtracurricularActivities,PlacementTraining,SSC_Marks,HSC_Marks,PlacementStatus
0,1,7.5,1,1,1,65,4.4,No,No,61,79,NotPlaced
1,2,8.9,0,3,2,90,4.0,Yes,Yes,78,82,Placed
2,3,7.3,1,2,2,82,4.8,Yes,No,79,80,NotPlaced
3,4,7.5,1,1,2,85,4.4,Yes,Yes,81,80,Placed
4,5,8.3,1,2,2,86,4.5,Yes,Yes,74,88,Placed


## Basic Cleaning

In [4]:
#  CLEAN TEXT
df["PlacementStatus"] = df["PlacementStatus"].str.strip().str.lower()

#  FIXED MAPPING
df["PlacementStatus"] = df["PlacementStatus"].map({
    "not placed": 0,
    "placed": 1
})

#  REMOVE NaN
df = df.dropna()

## Outliers

In [5]:
def remove_outliers(df):
    for col in df.select_dtypes(include=np.number).columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df = df[(df[col] >= lower) & (df[col] <= upper)]
    return df

df = remove_outliers(df)

## Encoder

In [6]:
# Target (FIXED MAPPING)
df["PlacementStatus"] = df["PlacementStatus"].map({
    "Not Placed": 0,
    "Placed": 1
})

# Features (ye rehne de — sahi hai)
from sklearn.preprocessing import LabelEncoder

df["ExtracurricularActivities"] = LabelEncoder().fit_transform(df["ExtracurricularActivities"])
df["PlacementTraining"] = LabelEncoder().fit_transform(df["PlacementTraining"])

In [7]:
df["PlacementTraining"]

1       1
3       1
4       1
9       1
10      1
       ..
9989    0
9991    1
9992    1
9997    1
9998    1
Name: PlacementTraining, Length: 3281, dtype: int64

## Features && Target

In [17]:
X = df.drop("PlacementStatus", axis=1)
y = df["PlacementStatus"]
print(y)

1      NaN
3      NaN
4      NaN
9      NaN
10     NaN
        ..
9989   NaN
9991   NaN
9992   NaN
9997   NaN
9998   NaN
Name: PlacementStatus, Length: 3281, dtype: float64


## Train-Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Scaling

In [10]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [16]:
print(df["PlacementStatus"].unique())
print(y.isnull().sum())

[nan]
3281


In [12]:
print(X.dtypes)

StudentID                      int64
CGPA                         float64
Internships                    int64
Projects                       int64
Workshops/Certifications       int64
AptitudeTestScore              int64
SoftSkillsRating             float64
ExtracurricularActivities      int64
PlacementTraining              int64
SSC_Marks                      int64
HSC_Marks                      int64
dtype: object


## Model Train

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)


model = LogisticRegression()
model.fit(X_train, y_train)

ValueError: Input y contains NaN.

## Evaluation

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8003629764065335

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.87      0.86       744
           1       0.71      0.65      0.68       358

    accuracy                           0.80      1102
   macro avg       0.77      0.76      0.77      1102
weighted avg       0.80      0.80      0.80      1102


Confusion Matrix:
 [[649  95]
 [125 233]]


## Sample

In [ ]:
sample = pd.DataFrame([[8.5, 2, 4, 2, 85, 4.5, 2, 1, 85, 90]],
                      columns=X.columns)

sample = scaler.transform(sample)

print("Prediction:", model.predict(sample))
print("Probability:", model.predict_proba(sample))

Prediction: [1]
Probability: [[0.05314322 0.94685678]]


## Save Model + Scaler

In [ ]:
pickle.dump(model, open("../model/placement_model.pkl", "wb"))
pickle.dump(scaler, open("../model/scaler.pkl", "wb"))

In [ ]:
print(X.columns)

Index(['CGPA', 'Internships', 'Projects', 'Workshops/Certifications',
       'AptitudeTestScore', 'SoftSkillsRating', 'ExtracurricularActivities',
       'PlacementTraining', 'SSC_Marks', 'HSC_Marks'],
      dtype='object')


In [ ]:
print(model.classes_)
print(model.predict(sample))
print(model.predict_proba(sample))

[0 1]
[1]
[[0.05314322 0.94685678]]


In [ ]:
print()